# Running Models — Audio · Part 1 — Companion Notebook

This goes with **"Audio Is a Tensor Too: Waveforms, Sample Rate, and Spectrograms"**. Run cells top to bottom (Shift+Enter). No GPU needed for this one — it's all CPU-friendly.

## Install

`torchaudio` reads audio into tensors; `soundfile` is the backend that decodes the WAV.

In [ ]:
!pip install -q torchaudio soundfile

## Load a real clip

Pull down a short speech clip from torchaudio's tutorial assets and read its shape, rate, and duration. Shape is `[channels, samples]` — channels first, like image color channels, but 1-D in time.

In [ ]:
import torchaudio

# a short speech clip bundled with torchaudio's tutorial assets
path = torchaudio.utils.download_asset(
    "tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav"
)
waveform, sample_rate = torchaudio.load(path)

print(waveform.shape)                          # torch.Size([1, 54400]) -> [channels, samples]
print("sample rate:", sample_rate)             # 16000
print("duration (s):", waveform.shape[1] / sample_rate)

## Hear it

Play the tensor back. The `rate=` argument matters as much as the samples — wrong rate, wrong speed.

In [ ]:
from IPython.display import Audio
Audio(waveform.numpy(), rate=sample_rate)

## Mono vs stereo

Most music is stereo (two channels). Models usually want one. Average the channels to downmix; `keepdim=True` keeps the channels-first `[1, samples]` shape. (This clip is already mono, so this is a no-op here.)

In [ ]:
mono = waveform.mean(dim=0, keepdim=True)      # [1, samples]

## Plot the waveform

Amplitude over time — the literal shape of the sound.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 2))
plt.plot(waveform[0].numpy())
plt.title("waveform: amplitude over time")
plt.show()

## Sample rate matters — resample

The sample rate is part of the data. A model trained at one rate expects exactly that rate; resample to match.

In [ ]:
resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=8000)
downsampled = resampler(waveform)
print(downsampled.shape)                        # fewer samples at 8 kHz

## The spectrogram — audio's image view

Instead of amplitude over time, plot how much of each frequency is present over time. That 2-D grid is basically an image — which is why many "audio" models are really vision/transformer models eating spectrograms.

In [ ]:
mel = torchaudio.transforms.MelSpectrogram(sample_rate=sample_rate, n_mels=64)
spec = mel(waveform)                            # [channels, n_mels, time]
print(spec.shape)                               # torch.Size([1, 64, 273])

plt.figure(figsize=(10, 3))
plt.imshow(spec[0].log2().clamp(min=-10), origin="lower", aspect="auto")
plt.title("mel spectrogram: frequency (y) over time (x)")
plt.show()

**The point:** a spectrogram is a 2-D grid = an image. A model that classifies sounds can literally be a vision model looking at spectrograms — the bridge from the Phase I vision/transformer posts into audio.